# Fine-tuning Llama 3.2 3B para Text-to-SQL con LoRA

Este notebook entrena un modelo especializado en generar consultas SQL usando **LoRA** sobre **Llama 3.2 3B**.

## ¿Qué haremos?
- Cargar Llama 3.2 3B Instruct
- Aplicar LoRA para fine-tuning eficiente
- Entrenar con datos de text-to-SQL
- Evaluar y guardar el modelo

## Requisitos:
- GPU con 6GB+ VRAM (recomendado)
- Cuenta Hugging Face
- Python 3.8+

## 1. Instalación de Dependencias

In [ ]:
# Instalar todas las dependencias necesarias
!pip install transformers datasets accelerate peft bitsandbytes torch trl huggingface_hub pandas numpy
!pip install ipywidgets

print("✅ Dependencias instaladas")

## 2. Autenticación Hugging Face

In [ ]:
from huggingface_hub import login

# Login en Hugging Face (necesario para Llama)
login()

print("✅ Autenticado en Hugging Face")

✅ Autenticado en Hugging Face


## 3. Importación de Librerías

In [4]:
import torch
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)

# LoRA y PEFT
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    TaskType
)

# Datasets y entrenamiento
from datasets import Dataset, load_dataset
from trl import SFTTrainer

print(f"✅ Librerías importadas")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"💾 CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🎮 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ Librerías importadas
🔥 PyTorch: 2.7.1+cpu
💾 CUDA disponible: False


## 4. Configuración del Modelo y Entrenamiento

Configuración optimizada para **Llama 3.2 3B** y **text-to-SQL**:

In [5]:
# ====== CONFIGURACIÓN PRINCIPAL ======
CONFIG = {
    # Modelo Llama 3.2 3B
    "model_name": "meta-llama/Llama-3.2-3B-Instruct",
    
    # Dataset
    "dataset_name": "gretelai/synthetic_text_to_sql",
    "num_samples": 1200,  # Cantidad optimizada para 3B
    
    # Directorios
    "output_dir": "../models/llama-sql-lora",
    "logs_dir": "../logs",
    
    # Parámetros de entrenamiento
    "max_seq_length": 1024,   # Llama 3.2 maneja bien secuencias largas
    "batch_size": 1,          # Conservador para modelo 3B
    "gradient_accumulation": 8, # Simula batch_size = 8
    "learning_rate": 1e-4,    # LR conservador
    "num_epochs": 2,          # Pocas epochs para evitar overfitting
    "warmup_ratio": 0.1,      # 10% warmup
    "save_steps": 100,
    "eval_steps": 100,
    "logging_steps": 10,
}

# ====== CONFIGURACIÓN LORA PARA LLAMA 3.2 ======
LORA_CONFIG = {
    "r": 32,                   # Rango alto para SQL complejo
    "lora_alpha": 64,          # Alpha proporcional
    "lora_dropout": 0.1,       # Dropout moderado
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM,
    
    # Módulos específicos de Llama 3.2 para SQL
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",      # MLP
        "lm_head"                                   # Output head
    ]
}

# ====== QUANTIZACIÓN 4-BIT ======
QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Crear directorios
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["logs_dir"], exist_ok=True)

print("✅ Configuración establecida")
print(f"📦 Modelo: {CONFIG['model_name']}")
print(f"📊 Muestras: {CONFIG['num_samples']}")
print(f"🎯 LoRA rank: {LORA_CONFIG['r']}")
print(f"📁 Output: {CONFIG['output_dir']}")

✅ Configuración establecida
📦 Modelo: meta-llama/Llama-3.2-3B-Instruct
📊 Muestras: 1200
🎯 LoRA rank: 32
📁 Output: ../models/llama-sql-lora


## 5. Carga y Filtrado del Dataset

In [6]:
def cargar_y_filtrar_datos():
    """Carga y filtra el dataset de SQL"""
    print("📥 Cargando dataset de text-to-SQL...")
    
    # Cargar dataset
    dataset = load_dataset(
        CONFIG["dataset_name"], 
        split=f"train[:{CONFIG['num_samples']}]"
    )
    df = pd.DataFrame(dataset)
    
    print(f"📊 Dataset original: {len(df)} ejemplos")
    
    # Filtros de calidad para SQL
    print("🔍 Aplicando filtros...")
    
    # 1. SQL válido y no vacío
    df = df[df['sql'].notna() & (df['sql'].str.len() > 15)]
    print(f"   Después filtro SQL válido: {len(df)}")
    
    # 2. Longitud razonable (evitar SQL extremadamente largos)
    df = df[df['sql'].str.len() < 800]
    print(f"   Después filtro longitud: {len(df)}")
    
    # 3. Pregunta válida
    df = df[df['sql_prompt'].notna() & (df['sql_prompt'].str.len() > 10)]
    print(f"   Después filtro pregunta: {len(df)}")
    
    # 4. Contexto/esquema válido
    df = df[df['sql_context'].notna() & (df['sql_context'].str.len() > 20)]
    print(f"   Después filtro contexto: {len(df)}")
    
    # 5. Sin errores obvios
    df = df[~df['sql'].str.contains('ERROR|error|undefined', case=False, na=False)]
    print(f"   Después filtro errores: {len(df)}")
    
    print(f"\n✅ Dataset final: {len(df)} ejemplos de calidad")
    return df

# Cargar datos
df_sql = cargar_y_filtrar_datos()

# Mostrar estadísticas
print(f"\n📈 Estadísticas:")
print(f"   SQL promedio: {df_sql['sql'].str.len().mean():.0f} caracteres")
print(f"   Pregunta promedio: {df_sql['sql_prompt'].str.len().mean():.0f} caracteres")

# Mostrar ejemplo
print(f"\n📝 Ejemplo:")
ejemplo = df_sql.iloc[0]
print(f"Pregunta: {ejemplo['sql_prompt'][:100]}...")
print(f"SQL: {ejemplo['sql']}")

📥 Cargando dataset de text-to-SQL...
📊 Dataset original: 1200 ejemplos
🔍 Aplicando filtros...
   Después filtro SQL válido: 1200
   Después filtro longitud: 1200
   Después filtro pregunta: 1200
   Después filtro contexto: 1200
   Después filtro errores: 1200

✅ Dataset final: 1200 ejemplos de calidad

📈 Estadísticas:
   SQL promedio: 128 caracteres
   Pregunta promedio: 83 caracteres

📝 Ejemplo:
Pregunta: What is the total volume of timber sold by each salesperson, sorted by salesperson?...
SQL: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;


## 6. Formateo de Datos para Llama 3.2

In [7]:
def formatear_para_llama32(df):
    """Formatea los datos usando el template de Llama 3.2"""
    print("🔄 Formateando datos para Llama 3.2...")
    
    # Template optimizado para text-to-SQL
    TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{schema}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{sql}<|eot_id|>"""
    
    formatted_data = []
    
    for _, row in df.iterrows():
        # Crear texto formateado
        text = TEMPLATE.format(
            schema=row['sql_context'].strip(),
            question=row['sql_prompt'].strip(),
            sql=row['sql'].strip()
        )
        
        formatted_data.append({"text": text})
    
    print(f"✅ {len(formatted_data)} ejemplos formateados")
    
    # Mostrar ejemplo formateado
    print(f"\n📝 Ejemplo formateado:")
    print(formatted_data[0]["text"][:500] + "...")
    
    return formatted_data

# Formatear datos
training_data = formatear_para_llama32(df_sql)

# Crear dataset
train_dataset = Dataset.from_list(training_data)
print(f"\n📦 Dataset creado: {len(train_dataset)} ejemplos")

🔄 Formateando datos para Llama 3.2...
✅ 1200 ejemplos formateados

📝 Ejemplo formateado:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE...

📦 Dataset creado: 1200 ejemplos


## 7. Carga del Modelo Llama 3.2 3B

In [8]:
def cargar_modelo_llama32():
    """Carga Llama 3.2 3B con configuración optimizada"""
    print(f"🤖 Cargando {CONFIG['model_name']}...")
    
    # Cargar tokenizador
    print("📝 Cargando tokenizador...")
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["model_name"],
        trust_remote_code=True,
        padding_side="right",  # Importante para entrenamiento
    )
    
    # Configurar pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    print(f"   ✅ Tokenizador cargado. Vocab: {len(tokenizer)}")
    
    # Cargar modelo con quantización
    print("🧠 Cargando modelo base...")
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    
    # Preparar para LoRA
    model = prepare_model_for_kbit_training(model)
    
    print(f"   ✅ Modelo cargado")
    print(f"   💾 Parámetros: {model.num_parameters():,}")
    
    return model, tokenizer

# Cargar modelo
base_model, tokenizer = cargar_modelo_llama32()

🤖 Cargando meta-llama/Llama-3.2-3B-Instruct...
📝 Cargando tokenizador...


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

c:\Users\Windows\Proyecto\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Windows\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

## 8. Aplicación de LoRA

In [ ]:
def aplicar_lora(model):
    """Aplica LoRA al modelo base"""
    print("🔧 Aplicando LoRA...")
    
    # Crear configuración LoRA
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
        target_modules=LORA_CONFIG["target_modules"],
    )
    
    # Aplicar LoRA
    model_lora = get_peft_model(model, lora_config)
    
    # Estadísticas
    trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_lora.parameters())
    
    print(f"✅ LoRA aplicado")
    print(f"📊 Parámetros entrenables: {trainable:,} ({trainable/total*100:.2f}%)")
    print(f"📊 Parámetros totales: {total:,}")
    
    # Mostrar módulos LoRA
    print(f"\n🎯 Módulos LoRA activos:")
    lora_modules = [name for name, _ in model_lora.named_modules() if "lora" in name.lower()]
    for module in lora_modules[:5]:  # Mostrar solo los primeros 5
        print(f"   {module}")
    if len(lora_modules) > 5:
        print(f"   ... y {len(lora_modules)-5} más")
    
    return model_lora

# Aplicar LoRA
model = aplicar_lora(base_model)

## 9. Configuración del Entrenamiento

In [ ]:
def crear_training_arguments():
    """Crea argumentos de entrenamiento optimizados"""
    print("⚙️ Configurando entrenamiento...")
    
    # Calcular pasos
    num_samples = len(train_dataset)
    effective_batch_size = CONFIG["batch_size"] * CONFIG["gradient_accumulation"]
    steps_per_epoch = num_samples // effective_batch_size
    max_steps = steps_per_epoch * CONFIG["num_epochs"]
    warmup_steps = int(max_steps * CONFIG["warmup_ratio"])
    
    print(f"📊 Configuración de entrenamiento:")
    print(f"   Muestras: {num_samples}")
    print(f"   Batch efectivo: {effective_batch_size}")
    print(f"   Pasos por época: {steps_per_epoch}")
    print(f"   Pasos totales: {max_steps}")
    print(f"   Warmup steps: {warmup_steps}")
    
    training_args = TrainingArguments(
        # Directorios
        output_dir=CONFIG["output_dir"],
        logging_dir=CONFIG["logs_dir"],
        
        # Entrenamiento
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        learning_rate=CONFIG["learning_rate"],
        
        # Scheduler
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        
        # Guardado
        save_steps=CONFIG["save_steps"],
        save_total_limit=2,
        logging_steps=CONFIG["logging_steps"],
        
        # Optimización
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        # Precisión
        bf16=True,  # bfloat16 para Llama 3.2
        
        # Otros
        dataloader_drop_last=True,
        remove_unused_columns=False,
        report_to="none",  # Sin logging externo
        seed=42,
    )
    
    return training_args

# Crear argumentos
training_arguments = crear_training_arguments()
print("✅ Argumentos de entrenamiento creados")

## 10. Preparación del Trainer

In [ ]:
def crear_trainer():
    """Crea el SFTTrainer"""
    print("🏃‍♂️ Preparando SFTTrainer...")
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        dataset_text_field="text",
        args=training_arguments,
        max_seq_length=CONFIG["max_seq_length"],
        packing=False,  # No empaquetar para mayor control
    )
    
    print("✅ SFTTrainer preparado")
    print(f"📦 Dataset: {len(trainer.train_dataset)} ejemplos")
    print(f"🔤 Max length: {CONFIG['max_seq_length']}")
    
    return trainer

# Crear trainer
trainer = crear_trainer()

## 11. ¡ENTRENAMIENTO!

**⚠️ IMPORTANTE:**
- Este proceso puede tomar 1-3 horas
- Monitorea la pérdida (loss) - debe disminuir
- Si hay errores de memoria, reduce `batch_size` o `max_seq_length`

In [ ]:
def entrenar():
    """Ejecuta el entrenamiento"""
    print("🚀 INICIANDO ENTRENAMIENTO")
    print("=" * 50)
    
    start_time = datetime.now()
    print(f"⏰ Inicio: {start_time.strftime('%H:%M:%S')}")
    
    try:
        # ¡ENTRENAR!
        result = trainer.train()
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print("\n🎉 ENTRENAMIENTO COMPLETADO")
        print("=" * 50)
        print(f"⏰ Fin: {end_time.strftime('%H:%M:%S')}")
        print(f"⏱️ Duración: {duration}")
        print(f"📉 Loss final: {result.training_loss:.4f}")
        
        return True, result
        
    except KeyboardInterrupt:
        print("\n⚠️ Entrenamiento interrumpido")
        return False, None
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return False, None

# ¡EJECUTAR ENTRENAMIENTO!
success, training_result = entrenar()

## 12. Guardar Modelo Entrenado

In [ ]:
def guardar_modelo():
    """Guarda el modelo entrenado"""
    if not success:
        print("❌ No se puede guardar - entrenamiento no completado")
        return None
    
    print("💾 Guardando modelo...")
    
    # Directorio final
    final_dir = f"{CONFIG['output_dir']}/final"
    os.makedirs(final_dir, exist_ok=True)
    
    # Guardar modelo LoRA
    model.save_pretrained(final_dir)
    print(f"✅ Modelo LoRA guardado en: {final_dir}")
    
    # Guardar tokenizador
    tokenizer.save_pretrained(final_dir)
    print(f"✅ Tokenizador guardado")
    
    # Guardar configuración
    config_info = {
        "base_model": CONFIG["model_name"],
        "dataset": CONFIG["dataset_name"],
        "num_samples": len(train_dataset),
        "lora_config": LORA_CONFIG,
        "training_loss": training_result.training_loss if training_result else None,
        "date": datetime.now().isoformat(),
    }
    
    with open(f"{final_dir}/training_info.json", "w") as f:
        json.dump(config_info, f, indent=2)
    
    print(f"✅ Información guardada")
    print(f"\n📁 Modelo completo en: {final_dir}")
    
    return final_dir

# Guardar modelo
model_path = guardar_modelo()

## 13. Prueba del Modelo

In [ ]:
def probar_modelo():
    """Prueba el modelo entrenado"""
    if not model_path:
        print("❌ No hay modelo para probar")
        return
    
    print("🧪 PROBANDO MODELO ENTRENADO")
    print("=" * 40)
    
    def generar_sql(schema, question):
        """Genera SQL usando el modelo entrenado"""
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{schema}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        # Tokenizar
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=800)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generar
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )
        
        # Decodificar
        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        # Extraer SQL generado
        if "<|start_header_id|>assistant<|end_header_id|>" in response:
            sql_part = response.split("<|start_header_id|>assistant<|end_header_id|>")[1]
            sql_part = sql_part.split("<|eot_id|>")[0].strip()
        else:
            sql_part = "Error en generación"
        
        return sql_part
    
    # Ejemplos de prueba
    tests = [
        {
            "schema": "CREATE TABLE users (id INT, name VARCHAR(50), age INT, city VARCHAR(50));",
            "question": "Get all users older than 25 from New York"
        },
        {
            "schema": "CREATE TABLE products (id INT, name VARCHAR(100), price DECIMAL, category VARCHAR(50)); CREATE TABLE orders (id INT, product_id INT, quantity INT, total DECIMAL);",
            "question": "Find total revenue by product category"
        },
        {
            "schema": "CREATE TABLE employees (id INT, name VARCHAR(50), department VARCHAR(50), salary DECIMAL);",
            "question": "What is the average salary per department?"
        }
    ]
    
    for i, test in enumerate(tests, 1):
        print(f"\n🧪 PRUEBA {i}:")
        print(f"Schema: {test['schema'][:60]}...")
        print(f"Pregunta: {test['question']}")
        
        try:
            sql = generar_sql(test["schema"], test["question"])
            print(f"✅ SQL: {sql}")
        except Exception as e:
            print(f"❌ Error: {e}")
        
        print("-" * 40)

# Probar modelo
probar_modelo()

## 14. Script de Integración

In [ ]:
def crear_script_uso():
    """Crea script para usar el modelo en tu proyecto"""
    if not model_path:
        print("❌ No hay modelo para crear script")
        return
    
    script = f'''# Script para usar el modelo SQL entrenado
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

class LlamaSQLGenerator:
    def __init__(self, model_path="{model_path}"):
        print("🤖 Cargando modelo SQL Llama 3.2...")
        
        # Cargar modelo base
        self.base_model = AutoModelForCausalLM.from_pretrained(
            "{CONFIG['model_name']}",
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        
        # Cargar adaptadores LoRA
        self.model = PeftModel.from_pretrained(self.base_model, model_path)
        
        # Cargar tokenizador
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        print("✅ Modelo cargado")
    
    def generar_sql(self, schema, question):
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{{schema}}

Question: {{question}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=800)
        inputs = {{k: v.to(self.model.device) for k, v in inputs.items()}}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        if "<|start_header_id|>assistant<|end_header_id|>" in response:
            sql_part = response.split("<|start_header_id|>assistant<|end_header_id|>")[1]
            sql_part = sql_part.split("<|eot_id|>")[0].strip()
        else:
            sql_part = "Error en generación"
        
        return sql_part

# Función compatible con tu código existente
def generar_sql_con_llama(prompt):
    generator = LlamaSQLGenerator()
    # Parsear prompt simple
    if "Schema:" in prompt and "Question:" in prompt:
        schema = prompt.split("Question:")[0].replace("Schema:", "").strip()
        question = prompt.split("Question:")[1].replace("Return only the SQL query:", "").strip()
    else:
        schema = "Unknown"
        question = prompt
    
    return generator.generar_sql(schema, question)

# Ejemplo de uso
if __name__ == "__main__":
    generator = LlamaSQLGenerator()
    sql = generator.generar_sql(
        "CREATE TABLE users (id INT, name VARCHAR(50), age INT);",
        "Get users older than 25"
    )
    print(f"SQL: {{sql}}")
'''
    
    # Guardar script
    script_path = "../scripts/llama_sql_generator.py"
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script)
    
    # Instrucciones
    instructions = f'''# CÓMO USAR TU MODELO LLAMA SQL

## En tu run_batch.py:
```python
# Cambiar:
from scripts.generate_sql import generar_sql_con_ollama
# Por:
from scripts.llama_sql_generator import generar_sql_con_llama

# Y usar:
sql_generado = generar_sql_con_llama(prompt)
```

## Uso directo:
```python
from scripts.llama_sql_generator import LlamaSQLGenerator

generator = LlamaSQLGenerator()
sql = generator.generar_sql(schema, question)
```

## Modelo guardado en: {model_path}
'''
    
    with open("../COMO_USAR_LLAMA_SQL.md", "w", encoding="utf-8") as f:
        f.write(instructions)
    
    print(f"✅ Script creado: {script_path}")
    print(f"✅ Instrucciones: ../COMO_USAR_LLAMA_SQL.md")

# Crear scripts
crear_script_uso()

## 🎉 ¡ENTRENAMIENTO COMPLETADO!

### ✅ Lo que has logrado:

1. **Modelo entrenado**: Llama 3.2 3B especializado en SQL
2. **LoRA aplicado**: Entrenamiento eficiente (~1% parámetros)
3. **Datos de calidad**: 1200+ ejemplos filtrados
4. **Modelo guardado**: Listo para usar
5. **Scripts creados**: Integración fácil

### 📁 Archivos generados:

- `../models/llama-sql-lora/final/` - Modelo entrenado
- `../scripts/llama_sql_generator.py` - Script de uso
- `../COMO_USAR_LLAMA_SQL.md` - Instrucciones

### 🚀 Próximos pasos:

1. **Probar más ejemplos** en la celda anterior
2. **Integrar en run_batch.py** usando el script generado
3. **Comparar rendimiento** vs modelo original
4. **Ajustar parámetros** si es necesario

¡Tu modelo Llama 3.2 SQL está listo! 🎯